Nested Models

In [1]:
from pydantic import BaseModel, Field, EmailStr, AnyUrl, field_validator, model_validator, computed_field
from typing import List, Dict, Optional, Annotated

In [2]:
class Patient(BaseModel):
    name: str = Annotated[str, Field(max_length=50, title="Patient Name", description="Enter name of the patient in less than 50 characters", examples=["John Doe", "Jane Smith"])]
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=120)
    weight: Annotated[float, Field(gt=0, strict=True)]
    height: Annotated[float, Field(gt=0, strict=True)]
    married: Annotated[Optional[bool], Field(default=False, description="Is the patient married?")]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=2)]
    contact_info: Dict[str, str]

    @field_validator('email')
    @classmethod
    def validate_email(cls, value):
        allowed_domains = ['sbi.com', 'equitas.com']
        domain = value.split('@')[-1]
        if domain not in allowed_domains:
            raise ValueError(f"Email domain must be one of {allowed_domains}")
        return value
    
    @field_validator('name', mode='after')
    @classmethod
    def transform_name(cls, value):
        return value.strip().upper()
    
    @field_validator('age')
    @classmethod
    def validate_age(cls, value):
        if value < 18:
            raise ValueError("Patient must be at least 18 years old")
        return value
    
    @model_validator(mode='after')
    def validate_emergency_contact(cls, model):
        if model.age > 60 and 'emergency_contact' not in model.contact_info:
            raise ValueError("Patients over 60 must have an emergency contact")
        return model
    
    @computed_field
    @property
    def bmi(self) -> float:
        return round(self.weight / ((self.height / 100) ** 2), 2)

In [3]:
patient_info = {'name': 'benky', 'email': 'benky@sbi.com', 'linkedin_url': 'https://linkedin.com/in/benky', 'age': '65', 'weight': 70.5, 'height': 175.0, 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890', 'emergency_contact': '9876543210'}}
patient1 = Patient(**patient_info)
print(patient1)

name='BENKY' email='benky@sbi.com' linkedin_url=AnyUrl('https://linkedin.com/in/benky') age=65 weight=70.5 height=175.0 married=False allergies=['dust', 'pollen'] contact_info={'phone': '1234567890', 'emergency_contact': '9876543210'} bmi=23.02


Lets add a new field address, but I don't want to keep it as a string, instead I want it as a seperate pydantic model such that I could extract state, pin etc

In [4]:
class Address(BaseModel):
    city: str
    state: str
    pin: str

In [5]:
address_info = {'city': 'New York', 'state': 'NY', 'pin': '10001'}
address1 = Address(**address_info)
print(address1)

city='New York' state='NY' pin='10001'


In [6]:
class Patient(BaseModel):
    name: str = Annotated[str, Field(max_length=50, title="Patient Name", description="Enter name of the patient in less than 50 characters", examples=["John Doe", "Jane Smith"])]
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=120)
    weight: Annotated[float, Field(gt=0, strict=True)]
    height: Annotated[float, Field(gt=0, strict=True)]
    married: Annotated[Optional[bool], Field(default=False, description="Is the patient married?")]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=2)]
    contact_info: Dict[str, str]
    address: Address

address field is coming as Address pydantic object

In [7]:
patient_info = {'name': 'benky', 'email': 'benky@sbi.com', 'linkedin_url': 'https://linkedin.com/in/benky', 'age': '65', 'weight': 70.5, 'height': 175.0, 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890', 'emergency_contact': '9876543210'}, 'address': address1}
patient2 = Patient(**patient_info)
print(patient2)

name='benky' email='benky@sbi.com' linkedin_url=AnyUrl('https://linkedin.com/in/benky') age=65 weight=70.5 height=175.0 married=False allergies=['dust', 'pollen'] contact_info={'phone': '1234567890', 'emergency_contact': '9876543210'} address=Address(city='New York', state='NY', pin='10001')


Now I can extract whatever I want from the address

In [8]:
print(patient2.address)
print(patient2.address.city)
print(patient2.address.pin)

city='New York' state='NY' pin='10001'
New York
10001
